# SOPR Capitulation + Momentum Filter

**Problem:** The SOPR double capitulation signal has good entry timing, but suffers during prolonged downtrends (catching falling knives in bear markets).

**Solution:** Add a momentum filter - only take the signal when the broader trend is UP.

## Momentum Filters to Test
1. **Price > 50 MA** - Short-term uptrend
2. **Price > 200 MA** - Long-term uptrend (bull market)
3. **50 MA > 200 MA** - Golden cross (trend confirmation)
4. **Price > 20 MA** - Very short-term trend
5. **Combinations** - e.g., Price > 50 MA AND 50 MA > 200 MA

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Ready!")

In [ ]:
# Load data
DATA_DIR = Path("../data/raw")

sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")

df = sopr.join(sopr_sth, how='inner').join(price, how='inner').sort_index()
df = df[df.index >= '2018-12-15']  # 2019+

close = df['price']
print(f"Data: {len(df)} rows, {df.index.min().date()} to {df.index.max().date()}")

In [ ]:
# Calculate moving averages
df['ma_20'] = df['price'].rolling(20).mean()
df['ma_50'] = df['price'].rolling(50).mean()
df['ma_100'] = df['price'].rolling(100).mean()
df['ma_200'] = df['price'].rolling(200).mean()

# Create momentum conditions
df['price_above_20'] = df['price'] > df['ma_20']
df['price_above_50'] = df['price'] > df['ma_50']
df['price_above_100'] = df['price'] > df['ma_100']
df['price_above_200'] = df['price'] > df['ma_200']
df['ma50_above_200'] = df['ma_50'] > df['ma_200']  # Golden cross state

print("Momentum conditions created:")
print(f"  Price > 20 MA: {df['price_above_20'].mean()*100:.1f}% of days")
print(f"  Price > 50 MA: {df['price_above_50'].mean()*100:.1f}% of days")
print(f"  Price > 100 MA: {df['price_above_100'].mean()*100:.1f}% of days")
print(f"  Price > 200 MA: {df['price_above_200'].mean()*100:.1f}% of days")
print(f"  50 MA > 200 MA: {df['ma50_above_200'].mean()*100:.1f}% of days")

In [ ]:
# Base entry signal (SOPR double capitulation)
both_below_1 = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
base_entries = both_below_1 & ~both_below_1.shift(1).fillna(False)

print(f"\nBase signal (no filter): {base_entries.sum()} entries")

---
## Visualize: Where Did We Lose Money?

Let's see which trades would have been filtered out by momentum.

In [ ]:
# Plot price with MAs and entry signals
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3],
                    subplot_titles=['Price with Moving Averages', 'SOPR Metrics'])

# Price and MAs
fig.add_trace(go.Scatter(x=df.index, y=df['price'], name='Price', line=dict(color='blue', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['ma_50'], name='50 MA', line=dict(color='orange', width=1, dash='dash')), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['ma_200'], name='200 MA', line=dict(color='red', width=1, dash='dash')), row=1, col=1)

# Entry signals - color by whether momentum filter would allow
entry_dates = base_entries[base_entries].index
for date in entry_dates:
    if date in df.index:
        above_200 = df.loc[date, 'price_above_200']
        color = 'green' if above_200 else 'red'
        symbol = 'triangle-up' if above_200 else 'x'
        fig.add_trace(go.Scatter(
            x=[date], y=[df.loc[date, 'price']],
            mode='markers',
            marker=dict(symbol=symbol, size=10, color=color),
            showlegend=False,
            hovertemplate=f"{date.strftime('%Y-%m-%d')}<br>Above 200MA: {above_200}<extra></extra>"
        ), row=1, col=1)

# SOPR
fig.add_trace(go.Scatter(x=df.index, y=df['sopr'], name='SOPR', line=dict(color='purple', width=1)), row=2, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='gray', row=2, col=1)

fig.update_layout(height=700, title_text='Entry Signals: Green = Above 200 MA (TAKE), Red X = Below 200 MA (SKIP)')
fig.update_yaxes(type='log', row=1, col=1)
fig.show()

---
## Trailing Stop Backtester (from previous notebook)

In [ ]:
def backtest_trailing_stop(
    close: pd.Series,
    entries: pd.Series,
    stop_loss: float = 0.08,
    trailing_stop: float = 0.12,
    min_profit_to_trail: float = 0.05,
    max_hold_days: int = 180,
):
    """Backtest with trailing stop."""
    trades = []
    entry_indices = entries[entries].index.tolist()
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = close.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        peak_price = entry_price
        initial_stop = entry_price * (1 - stop_loss)
        current_stop = initial_stop
        is_trailing = False
        
        exit_date = None
        exit_price = None
        exit_reason = None
        peak_reached = 0
        
        for j in range(entry_idx + 1, len(close)):
            current_date = close.index[j]
            current_price = close.iloc[j]
            days_held = j - entry_idx
            
            if current_price > peak_price:
                peak_price = current_price
                peak_reached = (peak_price - entry_price) / entry_price
            
            current_pnl = (current_price - entry_price) / entry_price
            
            if not is_trailing and current_pnl >= min_profit_to_trail:
                is_trailing = True
            
            if is_trailing:
                trailing_stop_level = peak_price * (1 - trailing_stop)
                if trailing_stop_level > current_stop:
                    current_stop = trailing_stop_level
            
            if current_price <= current_stop:
                exit_date = current_date
                exit_price = current_stop
                exit_reason = 'trailing_stop' if is_trailing else 'stop_loss'
                break
            
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = f'max_hold'
                break
        
        if exit_date is None:
            exit_date = close.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({
            'entry_date': entry_date,
            'entry_price': entry_price,
            'exit_date': exit_date,
            'exit_price': exit_price,
            'pnl_pct': pnl,
            'peak_pnl_pct': peak_reached,
            'days_held': (exit_date - entry_date).days,
            'exit_reason': exit_reason
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

---
## Test Momentum Filters

In [ ]:
# Define filter configurations
filters = {
    'No Filter (baseline)': pd.Series(True, index=df.index),
    'Price > 20 MA': df['price_above_20'],
    'Price > 50 MA': df['price_above_50'],
    'Price > 100 MA': df['price_above_100'],
    'Price > 200 MA': df['price_above_200'],
    '50 MA > 200 MA': df['ma50_above_200'],
    'Price > 50 AND 50 > 200': df['price_above_50'] & df['ma50_above_200'],
    'Price > 200 AND 50 > 200': df['price_above_200'] & df['ma50_above_200'],
    'Price > 50 OR Price > 200': df['price_above_50'] | df['price_above_200'],
}

In [ ]:
# Test each filter
results = []

# Best trailing stop params from previous notebook
STOP_LOSS = 0.08
TRAILING_STOP = 0.12
MIN_PROFIT = 0.05

for filter_name, filter_mask in filters.items():
    # Apply filter to entries
    filtered_entries = base_entries & filter_mask.reindex(base_entries.index, fill_value=False)
    
    # Run backtest
    trades = backtest_trailing_stop(
        close=close,
        entries=filtered_entries,
        stop_loss=STOP_LOSS,
        trailing_stop=TRAILING_STOP,
        min_profit_to_trail=MIN_PROFIT,
        max_hold_days=180
    )
    
    if len(trades) > 0:
        total_return = (1 + trades['pnl_pct']).prod() - 1
        win_rate = (trades['pnl_pct'] > 0).mean()
        avg_win = trades[trades['pnl_pct'] > 0]['pnl_pct'].mean() if (trades['pnl_pct'] > 0).any() else 0
        avg_loss = trades[trades['pnl_pct'] <= 0]['pnl_pct'].mean() if (trades['pnl_pct'] <= 0).any() else 0
        
        gross_win = trades[trades['pnl_pct'] > 0]['pnl_pct'].sum()
        gross_loss = abs(trades[trades['pnl_pct'] <= 0]['pnl_pct'].sum())
        profit_factor = gross_win / gross_loss if gross_loss > 0 else np.inf
        
        # Count filtered out trades
        filtered_out = base_entries.sum() - filtered_entries.sum()
        
        results.append({
            'filter': filter_name,
            'n_trades': len(trades),
            'filtered_out': filtered_out,
            'total_return': total_return,
            'win_rate': win_rate,
            'avg_win': avg_win,
            'avg_loss': avg_loss,
            'profit_factor': profit_factor,
            'avg_days': trades['days_held'].mean()
        })
    else:
        results.append({
            'filter': filter_name,
            'n_trades': 0,
            'filtered_out': base_entries.sum(),
            'total_return': 0,
            'win_rate': 0,
            'avg_win': 0,
            'avg_loss': 0,
            'profit_factor': 0,
            'avg_days': 0
        })

results_df = pd.DataFrame(results).sort_values('total_return', ascending=False)

print("MOMENTUM FILTER COMPARISON")
print("="*120)
print(results_df.to_string(index=False))

In [ ]:
# Visualize
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=['Total Return', 'Win Rate', 'Profit Factor', 'Number of Trades'])

plot_df = results_df.sort_values('total_return', ascending=True)
colors = ['green' if x > results_df[results_df['filter'] == 'No Filter (baseline)']['total_return'].values[0] else 'steelblue' 
          for x in plot_df['total_return']]

fig.add_trace(go.Bar(y=plot_df['filter'], x=plot_df['total_return']*100,
                     orientation='h', marker_color=colors), row=1, col=1)
fig.add_trace(go.Bar(y=plot_df['filter'], x=plot_df['win_rate']*100,
                     orientation='h', marker_color='steelblue'), row=1, col=2)
fig.add_trace(go.Bar(y=plot_df['filter'], x=plot_df['profit_factor'].clip(upper=10),
                     orientation='h', marker_color='purple'), row=2, col=1)
fig.add_trace(go.Bar(y=plot_df['filter'], x=plot_df['n_trades'],
                     orientation='h', marker_color='orange'), row=2, col=2)

# Add baseline reference
baseline_return = results_df[results_df['filter'] == 'No Filter (baseline)']['total_return'].values[0] * 100
fig.add_vline(x=baseline_return, line_dash='dash', line_color='red', row=1, col=1)

fig.update_layout(height=800, showlegend=False, 
                  title_text='Momentum Filters vs Baseline (Red line = No Filter)')
fig.show()

In [ ]:
# Best filter analysis
best = results_df.iloc[0]
baseline = results_df[results_df['filter'] == 'No Filter (baseline)'].iloc[0]

print(f"\n{'='*60}")
print(f"BEST FILTER: {best['filter']}")
print(f"{'='*60}")
print(f"\n{'Metric':<25} {'With Filter':>15} {'No Filter':>15} {'Change':>15}")
print("-"*75)
print(f"{'Trades':<25} {best['n_trades']:>15} {baseline['n_trades']:>15} {best['n_trades'] - baseline['n_trades']:>+15}")
print(f"{'Total Return':<25} {best['total_return']*100:>14.0f}% {baseline['total_return']*100:>14.0f}% {(best['total_return'] - baseline['total_return'])*100:>+14.0f}%")
print(f"{'Win Rate':<25} {best['win_rate']*100:>14.0f}% {baseline['win_rate']*100:>14.0f}% {(best['win_rate'] - baseline['win_rate'])*100:>+14.0f}%")
print(f"{'Avg Win':<25} {best['avg_win']*100:>14.1f}% {baseline['avg_win']*100:>14.1f}% {(best['avg_win'] - baseline['avg_win'])*100:>+14.1f}%")
print(f"{'Avg Loss':<25} {best['avg_loss']*100:>14.1f}% {baseline['avg_loss']*100:>14.1f}% {(best['avg_loss'] - baseline['avg_loss'])*100:>+14.1f}%")
print(f"{'Profit Factor':<25} {best['profit_factor']:>15.2f} {baseline['profit_factor']:>15.2f} {best['profit_factor'] - baseline['profit_factor']:>+15.2f}")

---
## Walk-Forward Validation

In [ ]:
def walk_forward_test(entries, filter_mask=None, train_days=365, test_days=90, step_days=90):
    """Walk-forward validation."""
    wf_results = []
    
    # Apply filter if provided
    if filter_mask is not None:
        filtered_entries = entries & filter_mask.reindex(entries.index, fill_value=False)
    else:
        filtered_entries = entries
    
    total_days = len(close)
    n_folds = (total_days - train_days) // step_days
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        if test_end <= test_start:
            break
        
        test_close = close.iloc[test_start:test_end]
        test_entries = filtered_entries.iloc[test_start:test_end]
        
        trades = backtest_trailing_stop(
            close=test_close,
            entries=test_entries,
            stop_loss=STOP_LOSS,
            trailing_stop=TRAILING_STOP,
            min_profit_to_trail=MIN_PROFIT,
            max_hold_days=180
        )
        
        if len(trades) > 0:
            strat_return = (1 + trades['pnl_pct']).prod() - 1
            n_trades = len(trades)
        else:
            strat_return = 0
            n_trades = 0
        
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        wf_results.append({
            'fold': fold,
            'period': close.index[test_start].strftime('%Y-%m'),
            'n_trades': n_trades,
            'strat_return': strat_return,
            'hold_return': hold_return,
            'excess': strat_return - hold_return,
            'beat_hold': strat_return > hold_return
        })
    
    return pd.DataFrame(wf_results)

In [ ]:
# Test walk-forward for top filters
top_filters = ['No Filter (baseline)', 'Price > 200 MA', 'Price > 50 MA', 
               '50 MA > 200 MA', 'Price > 50 AND 50 > 200']

wf_comparison = []

for filter_name in top_filters:
    if filter_name == 'No Filter (baseline)':
        filter_mask = None
    else:
        filter_mask = filters[filter_name]
    
    wf_df = walk_forward_test(base_entries, filter_mask)
    
    # Only count folds with trades
    wf_with_trades = wf_df[wf_df['n_trades'] > 0]
    
    wf_comparison.append({
        'filter': filter_name,
        'total_folds': len(wf_df),
        'folds_with_trades': len(wf_with_trades),
        'beat_hold_all': wf_df['beat_hold'].mean() if len(wf_df) > 0 else 0,
        'beat_hold_traded': wf_with_trades['beat_hold'].mean() if len(wf_with_trades) > 0 else 0,
        'avg_excess': wf_df['excess'].mean() if len(wf_df) > 0 else 0
    })
    
    print(f"\n{filter_name}:")
    print(f"  Beat B&H: {wf_df['beat_hold'].mean()*100:.0f}% | Avg Excess: {wf_df['excess'].mean()*100:+.1f}%")

wf_comp_df = pd.DataFrame(wf_comparison)
print("\n" + "="*80)
print("WALK-FORWARD COMPARISON")
print("="*80)
print(wf_comp_df.to_string(index=False))

In [ ]:
# Detailed walk-forward for best filter
best_filter_name = wf_comp_df.sort_values('beat_hold_all', ascending=False).iloc[0]['filter']
print(f"\nDetailed Walk-Forward for: {best_filter_name}")
print("="*80)

if best_filter_name == 'No Filter (baseline)':
    best_wf = walk_forward_test(base_entries, None)
else:
    best_wf = walk_forward_test(base_entries, filters[best_filter_name])

for _, row in best_wf.iterrows():
    status = '✓' if row['beat_hold'] else '✗'
    print(f"Fold {row['fold']:2d}: {row['period']} | "
          f"{row['n_trades']:2d} trades | "
          f"Strat: {row['strat_return']*100:+6.1f}% | "
          f"B&H: {row['hold_return']*100:+6.1f}% | {status}")

---
## Visualize Best Strategy

In [ ]:
# Run best strategy
if best_filter_name == 'No Filter (baseline)':
    best_entries = base_entries
else:
    best_entries = base_entries & filters[best_filter_name].reindex(base_entries.index, fill_value=False)

best_trades = backtest_trailing_stop(
    close=close,
    entries=best_entries,
    stop_loss=STOP_LOSS,
    trailing_stop=TRAILING_STOP,
    min_profit_to_trail=MIN_PROFIT,
    max_hold_days=180
)

print(f"Best Strategy: SOPR Capitulation + {best_filter_name}")
print(f"Trades: {len(best_trades)}")

In [ ]:
# Trade details
print("\nTRADE DETAILS")
print("="*100)
display_trades = best_trades.copy()
display_trades['entry_date'] = pd.to_datetime(display_trades['entry_date']).dt.strftime('%Y-%m-%d')
display_trades['exit_date'] = pd.to_datetime(display_trades['exit_date']).dt.strftime('%Y-%m-%d')
display_trades['entry_price'] = display_trades['entry_price'].round(0).astype(int)
display_trades['exit_price'] = display_trades['exit_price'].round(0).astype(int)
display_trades['pnl_pct'] = (display_trades['pnl_pct'] * 100).round(1)
display_trades['peak_pnl_pct'] = (display_trades['peak_pnl_pct'] * 100).round(1)

print(display_trades[['entry_date', 'entry_price', 'exit_date', 'exit_price',
                      'pnl_pct', 'peak_pnl_pct', 'days_held', 'exit_reason']].to_string(index=False))

In [ ]:
# All trades on chart
fig = go.Figure()

fig.add_trace(go.Scatter(x=close.index, y=close, name='BTC Price',
                         line=dict(color='lightblue', width=1)))

# Add MAs
fig.add_trace(go.Scatter(x=df.index, y=df['ma_200'], name='200 MA',
                         line=dict(color='red', width=1, dash='dash')))

for _, trade in best_trades.iterrows():
    color = 'green' if trade['pnl_pct'] > 0 else 'red'
    width = 3 if abs(trade['pnl_pct']) > 0.20 else 2
    
    fig.add_trace(go.Scatter(
        x=[trade['entry_date'], trade['exit_date']],
        y=[trade['entry_price'], trade['exit_price']],
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=8),
        showlegend=False,
        hovertemplate=f"Entry: {trade['entry_date'].strftime('%Y-%m-%d')}<br>" +
                      f"Exit: {trade['exit_date'].strftime('%Y-%m-%d')}<br>" +
                      f"PnL: {trade['pnl_pct']*100:.1f}%<br>" +
                      f"Exit: {trade['exit_reason']}<extra></extra>"
    ))

fig.update_layout(
    title=f'All Trades - SOPR + {best_filter_name}<br><sup>Green=Win, Red=Loss</sup>',
    yaxis_title='Price ($)',
    yaxis_type='log',
    height=600
)
fig.show()

In [ ]:
# Equity curve
equity = 100000 * (1 + best_trades['pnl_pct']).cumprod()
bh_final = 100000 * (close.iloc[-1] / close.iloc[0])

fig = go.Figure()
fig.add_trace(go.Scatter(x=best_trades['exit_date'], y=equity.values,
                         mode='lines+markers', name='Strategy'))
fig.add_hline(y=bh_final, line_dash='dash', line_color='gray',
              annotation_text=f'Buy & Hold: ${bh_final:,.0f}')
fig.add_hline(y=100000, line_dash='dot', line_color='black')

fig.update_layout(title=f'Equity Curve - SOPR + {best_filter_name}',
                  yaxis_title='Portfolio Value ($)', height=500)
fig.show()

print(f"\nFinal: ${equity.iloc[-1]:,.0f} vs B&H ${bh_final:,.0f}")

---
## Final Summary

In [ ]:
# Get stats for best filter
best_stats = results_df[results_df['filter'] == best_filter_name].iloc[0]
baseline_stats = results_df[results_df['filter'] == 'No Filter (baseline)'].iloc[0]
best_wf_stats = wf_comp_df[wf_comp_df['filter'] == best_filter_name].iloc[0]
baseline_wf_stats = wf_comp_df[wf_comp_df['filter'] == 'No Filter (baseline)'].iloc[0]

print("\n" + "="*70)
print("SOPR CAPITULATION + MOMENTUM FILTER - FINAL RESULTS")
print("="*70)

print(f"\n📊 STRATEGY")
print(f"   Entry: SOPR < 1 AND STH SOPR < 1")
print(f"   Filter: {best_filter_name}")
print(f"   Exit: Trailing Stop (SL {STOP_LOSS*100:.0f}%, Trail {TRAILING_STOP*100:.0f}%)")

print(f"\n📈 IN-SAMPLE COMPARISON")
print(f"   {'Metric':<20} {'With Filter':>15} {'No Filter':>15}")
print(f"   {'-'*55}")
print(f"   {'Trades':<20} {best_stats['n_trades']:>15} {baseline_stats['n_trades']:>15}")
print(f"   {'Total Return':<20} {best_stats['total_return']*100:>14.0f}% {baseline_stats['total_return']*100:>14.0f}%")
print(f"   {'Win Rate':<20} {best_stats['win_rate']*100:>14.0f}% {baseline_stats['win_rate']*100:>14.0f}%")
print(f"   {'Profit Factor':<20} {best_stats['profit_factor']:>15.2f} {baseline_stats['profit_factor']:>15.2f}")

print(f"\n🔍 WALK-FORWARD COMPARISON")
print(f"   {'Metric':<20} {'With Filter':>15} {'No Filter':>15}")
print(f"   {'-'*55}")
print(f"   {'Beat Buy&Hold':<20} {best_wf_stats['beat_hold_all']*100:>14.0f}% {baseline_wf_stats['beat_hold_all']*100:>14.0f}%")
print(f"   {'Avg Excess':<20} {best_wf_stats['avg_excess']*100:>+14.1f}% {baseline_wf_stats['avg_excess']*100:>+14.1f}%")

improvement = best_wf_stats['beat_hold_all'] - baseline_wf_stats['beat_hold_all']
if improvement > 0.05:
    verdict = f"✓ FILTER HELPS (+{improvement*100:.0f}% beat rate)"
elif improvement < -0.05:
    verdict = f"✗ FILTER HURTS ({improvement*100:.0f}% beat rate)"
else:
    verdict = "~ FILTER NEUTRAL"

print(f"\n🎯 VERDICT: {verdict}")
print("\n" + "="*70)

In [ ]:
# Save results
import json

final_results = {
    'signal': 'sopr_double_capitulation',
    'entry': 'SOPR < 1 AND STH_SOPR < 1',
    'best_momentum_filter': best_filter_name,
    'exit_strategy': 'trailing_stop',
    'exit_params': {
        'stop_loss': STOP_LOSS,
        'trailing_stop': TRAILING_STOP,
        'min_profit_to_trail': MIN_PROFIT
    },
    'with_filter': {
        'total_return': float(best_stats['total_return']),
        'win_rate': float(best_stats['win_rate']),
        'profit_factor': float(best_stats['profit_factor']),
        'n_trades': int(best_stats['n_trades']),
        'wf_beat_hold': float(best_wf_stats['beat_hold_all']),
        'wf_avg_excess': float(best_wf_stats['avg_excess'])
    },
    'without_filter': {
        'total_return': float(baseline_stats['total_return']),
        'win_rate': float(baseline_stats['win_rate']),
        'profit_factor': float(baseline_stats['profit_factor']),
        'n_trades': int(baseline_stats['n_trades']),
        'wf_beat_hold': float(baseline_wf_stats['beat_hold_all']),
        'wf_avg_excess': float(baseline_wf_stats['avg_excess'])
    },
    'all_filters_tested': results_df.to_dict('records')
}

with open('../data/sopr_momentum_filter_results.json', 'w') as f:
    json.dump(final_results, f, indent=2, default=str)

print("Saved to ../data/sopr_momentum_filter_results.json")